## Example 1: Parse a binary wheel's name

In [ ]:
import importlib.metadata

from wheel_filename import parse_wheel_filename

# Example 1: the PEP 427 fields of a platform wheel
assert importlib.metadata.version("wheel-filename") == "1.4.2"
w = parse_wheel_filename("numpy-2.3.0-cp314-cp314-manylinux_2_28_aarch64.whl")
assert (w.project, w.version, w.build) == ("numpy", "2.3.0", None), w
assert (w.python_tags, w.abi_tags, w.platform_tags) == (["cp314"], ["cp314"], ["manylinux_2_28_aarch64"]), w
assert str(w) == "numpy-2.3.0-cp314-cp314-manylinux_2_28_aarch64.whl"
print("Example 1 OK:", w.project, w.version, w.platform_tags)

## Example 2: Compressed tag sets and a build tag

In [ ]:
from wheel_filename import parse_wheel_filename

# Example 2: dotted tag sets expand into lists; the optional build tag sits after the version
w = parse_wheel_filename("pkg-1.0-1local-py2.py3-none-any.whl")
assert w.build == "1local" and w.python_tags == ["py2", "py3"], w
assert w.abi_tags == ["none"] and w.platform_tags == ["any"], w
multi = parse_wheel_filename("x-0.1-cp314-cp314-manylinux_2_17_aarch64.manylinux2014_aarch64.whl")
assert multi.platform_tags == ["manylinux_2_17_aarch64", "manylinux2014_aarch64"], multi.platform_tags
print("Example 2 OK:", w.build, multi.platform_tags)

## Example 3: A path works, the directory part is ignored

In [ ]:
from pathlib import Path

from wheel_filename import parse_wheel_filename

# Example 3: parse_wheel_filename takes str or os.PathLike and looks only at the base name
w = parse_wheel_filename(Path("/tmp/dist/requests-2.32.5-py3-none-any.whl"))
assert (w.project, w.version, w.python_tags) == ("requests", "2.32.5", ["py3"]), w
print("Example 3 OK:", w)

## Example 4: Invalid names raise InvalidFilenameError

In [ ]:
from wheel_filename import InvalidFilenameError, parse_wheel_filename

# Example 4: a sdist name, a missing tag and a wrong extension are all refused
for bad in ("numpy-2.3.0.tar.gz", "numpy-2.3.0-cp314-manylinux_2_28_aarch64.whl", "numpy-2.3.0-cp314-cp314-any.zip"):
    try:
        parse_wheel_filename(bad)
    except InvalidFilenameError as exc:
        assert exc.filename == bad, exc.filename
    else:
        raise AssertionError(f"accepted {bad!r}")
assert issubclass(InvalidFilenameError, ValueError)
print("Example 4 OK")

## Example 5: The wheel-filename command

In [ ]:
import json
import shutil
import subprocess

# Example 5: the console script prints the parsed fields as JSON
exe = shutil.which("wheel-filename")
assert exe, "wheel-filename is not on PATH"
out = subprocess.run([exe, "torch-2.9.0-cp314-cp314-linux_aarch64.whl"], capture_output=True, text=True, check=True)
fields = json.loads(out.stdout)
assert fields["project"] == "torch" and fields["abi_tags"] == ["cp314"], fields
print("Example 5 OK:", fields["platform_tags"])